# Core Numerical Methods Engine

## How Root Finding Works in Scientific Computing
In mathematics, a "root" of a function $f(x)$ is the value of $x$ where the 
function evaluates to zero:
$$
f(x) = 0
$$
In many real-world scientific problems, you cannot easily rearrange an equation 
with pen and paper to solve for $x$ algebraically because the equation is highly 
non-linear or relies on complex empirical lookup matrices.

A root-finding algorithm solves this by taking an initial guess for $x$ and 
iteratively evaluating the function, adjusting the guess step-by-step until 
$f(x)$ lands close enough to zero within an acceptable precision tolerance 
(e.g., $10^{-6}$).


## Applying Root-Finding to the Water Balance Equation

The core discrete water balance equation is:
$$S_{t+1} = S_t + R_t + I_t - ET_t - D_t$$

Let's assume we know our current soil state ($S_t$), today's rainfall ($R_t$), 
and today's weather inputs to calculate $ET_t$. We have a specific Target Soil 
Moisture ($S_{\text{target}}$) that we want to achieve tomorrow. We need to find 
the exact irrigation amount ($I_t$) that satisfies this goal.

The question now becomes:
> "How much irrigation water I must I apply today so that soil moisture S 
> reaches the target level $S_{\text{target}}$?"


To use a root-finding algorithm, we must rewrite this physical system as a 
function where our unknown variable is $x = I_t$, and our objective is to drive 
the difference between the simulated outcome and our target to zero:
$$f(I_t) = \text{Simulated } S_{t+1}(I_t) - S_{\text{target}} = 0$$

In [ ]:
# 1. Define the physical system model as our target function
def water_balance_objective(I_t, S_t, R_t, T, W, Solar, H, field_capacity, S_target):
    """
    Evaluates how far the resulting soil moisture is from our target 
    given a specific irrigation choice (I_t).
    Objective: Find I_t where this function returns 0.
    """
    # Calculate daily evapotranspiration
    et = max(0, 0.12*T + 0.35*W + 2.4*Solar - 0.025*H)
    
    # Core water balance transformation
    S_next = S_t + R_t + I_t - et
    
    # Apply physical drainage boundary ceiling
    if S_next > field_capacity:
        S_next = field_capacity
    if S_next < 0:
        S_next = 0
        
    # Return the residual error gap we want to eliminate
    return S_next - S_target



# 2. Implement the Secant Root-Finding Algorithm from scratch
def solve_irrigation_secant(S_t, R_t, T, W, Solar, H, field_capacity, S_target, max_iter=50, tol=1e-5):
    """
    Uses the Secant Method to find the precise root (I_t) for our objective function.
    """
    # Two initial guesses for irrigation (e.g., 0 mm and 10 mm)
    x0 = 0.0
    x1 = 30.0
    
    for iteration in range(max_iter):
        # Evaluate function at both guess points
        f_x0 = water_balance_objective(x0, S_t, R_t, T, W, Solar, H, field_capacity, S_target)
        f_x1 = water_balance_objective(x1, S_t, R_t, T, W, Solar, H, field_capacity, S_target)
        
        # Prevent division by zero if the function flattens out
        if abs(f_x1 - f_x0) < 1e-12:
            break
            
        # Secant update formula
        x_next = x1 - f_x1 * (x1 - x0) / (f_x1 - f_x0)

        # This prevents the algorithm from falling into the flat zero-moisture zone
        if x_next < 0:
            x_next = 0.0
        
        # Check if the solution has converged close enough to zero
        if abs(water_balance_objective(x_next, S_t, R_t, T, W, Solar, H, field_capacity, S_target)) < tol:
            # Physical constraint: You cannot apply negative irrigation
            return max(0.0, x_next)
        
        # Update guess bounds for the next step
        x0 = x1
        x1 = x_next
        
    return max(0.0, x1)


Running a multiday simulation loop

Import the dataset and do data normalization and zone filtering

In [ ]:
import pandas as pd

weather = pd.read_csv('../data/raw/weather_daily.csv', 
                            na_values=['NA', ''])
soil = pd.read_csv('../data/raw/soil_sensor_data.csv', 
                              na_values=['NA', ''])


weather = weather.dropna()
soil = soil.dropna()

weather["date"] = pd.to_datetime(weather["date"]).dt.date
soil["date"] = pd.to_datetime(soil["timestamp"]).dt.date

soil_zone_a = soil[soil["zone_id"] == "Zone_A"].copy()
 
# Drop the original timestamp & zone_id columns 
soil_zone_a = soil_zone_a.drop(columns=["timestamp", "zone_id"])
 
# Inner merge on date
merged = pd.merge(weather, soil_zone_a, on="date", how="inner")
 
# Sort & reset index
merged = merged.sort_values("date").reset_index(drop=True)

In [ ]:
# 1. Setup global structural constants
field_capacity = 50.0    # Ceiling of soil water storage (mm)
S_target = 38.5         # Our optimal structural moisture goal (mm)

# Initialize the starting soil moisture on Day 1 from your actual sensor data
# (Assuming your sensor column is named 'soil_moisture_pct' and is in percentage)
current_S = merged.loc[0, 'soil_moisture_pct'] * field_capacity / 100.0  # Convert percentage to mm

print(f"Initial soil moisture (S) on Day 1: {current_S:.2f} mm")

# Pre-allocate lists to hold our simulated results
calculated_irrigation = []
simulated_S_history = []

# 2. Step through the dataset day by day
for idx, row in merged.iterrows():
    
    # Extract today's environmental inputs from the current row
    R_today = row['rainfall_mm']
    T_today = row['temperature_c']
    W_today = row['wind_speed_mps']
    Solar_today = row['solar_index']
    H_today = row['humidity_pct']
    
    # 3. Call the root-finder to compute the exact irrigation needed for TODAY
    # It takes today's starting moisture (current_S) and finds the matching water volume
    opt_I = solve_irrigation_secant(
        current_S, R_today, T_today, W_today, Solar_today, H_today, field_capacity, S_target
    )
    
    # Record the computed irrigation choice
    calculated_irrigation.append(opt_I)
    
    # 4. Update the physics engine to calculate the true closing state of the day
    # We pass the optimal irrigation value back into the balance equation
    et_today = max(0, 0.12*T_today + 0.35*W_today + 2.4*Solar_today - 0.025*H_today)
    
    # Soil moisture calculation adding today's rain and calculated optimal water
    next_S = current_S + R_today + opt_I - et_today
    
    # Apply physical clipping boundaries
    if next_S > field_capacity:
        next_S = field_capacity
    if next_S < 0:
        next_S = 0
        
    # Record the ending soil storage state
    simulated_S_history.append(next_S)
    
    # CRITICAL STEP: Tomorrow's starting moisture is today's closing moisture
    current_S = next_S

# 5. Append the calculated arrays directly back into your main DataFrame
merged['Precision_Irrigation_Need'] = calculated_irrigation
merged['Simulated_Soil_State'] = simulated_S_history

# View the aggregated results
# Inspect the first few rows of the updated precision strategy
print(merged[['date', 'soil_moisture_pct', 'Precision_Irrigation_Need', 'Simulated_Soil_State']])

We loop sequentially instead of vectorizing the root-finder due to 
**temporal dependency (system memory)**. In Level 2, calculating 
evapotranspiration ($ET$) only required looking at that day's independent 
weather parameters. In Level 3, however, today's starting moisture state 
($S_t$) is explicitly defined by yesterday's final calculated moisture state 
($S_{t-1}$). Because the rows are linked in a chain, the computer must solve 
the roots sequentially day by day down the line.

### Root Finding Methods

#### 1. The Bisection Method

The Bisection Method is a **bracketing** or **closed** method used to find a 
real root of a continuous function $f(x) = 0$. It relies on the 
**Intermediate Value Theorem**, which states that if a continuous function 
changes signs over an interval $[a, b]$ (i.e., $f(a) \times f(b) < 0$), there 
must be at least one root $c$ within that interval.

1. **Initialization:** Choose a lower bound $a$ and an upper bound $b$ such 
that $f(a)$ and $f(b)$ have opposite signs.
2. **Midpoint Calculation:** Compute the midpoint of the interval:

$$c = \frac{a + b}{2}$$


3. **Evaluation:** Check if $c$ is the root (i.e., $f(c) = 0$ or within a tiny 
tolerance $\epsilon$).
4. **Interval Reduction:** * If $f(a) \times f(c) < 0$, the root lies in the 
lower half. Set $b = c$.
* If $f(b) \times f(c) < 0$, the root lies in the upper half. Set $a = c$.


5. **Iteration:** Repeat steps 2–4 until the interval width $|b - a|$ or 
$|f(c)|$ is smaller than your predefined tolerance.

* **Pros:** Guaranteed to converge if the function is continuous and bounds 
bracket a root.
* **Cons:** Converges slowly (linear convergence rate).


#### 2. The Newton-Raphson Method

The Newton-Raphson method is an **open** method, meaning it requires only a 
single initial guess $x_0$ rather than a bounded bracket. It utilizes calculus 
to approximate the curve linearly at each step.

1. **Initialization:** Start with an initial guess $x_0$ near the true root.
2. **Linear Approximation:** Draw a tangent line to the function graph at 
$(x_i, f(x_i))$. The slope of this line is the derivative $f'(x_i)$.
3. **Finding the Intercept:** Find where this tangent line crosses the x-axis. 
This x-intercept becomes the next, improved approximation $x_{i+1}$. 
Mathematically derived from the equation of a line:

$$x_{i+1} = x_i - \frac{f(x_i)}{f'(x_i)}$$


4. **Iteration:** Repeat the process substituting $x_{i+1}$ for $x_i$ until 
$|x_{i+1} - x_i|$ or $|f(x_{i+1})|$ drops below the tolerance limit.

* **Pros:** Extremely fast when it works (quadratic convergence rate—the number 
of correct decimal places roughly doubles with each iteration).
* **Cons:** Requires knowing the analytical derivative $f'(x)$. It can fail or 
diverge completely if $f'(x_i) \approx 0$ (a flat slope sends the next guess to 
infinity) or if the initial guess is too far from the root.

#### 3. The Secant Method

The Secant Method can be thought of as a variant of the Newton-Raphson method 
designed for scenarios where finding the analytical derivative $f'(x)$ is too 
difficult or computationally expensive. Instead of calculating a tangent line 
based on a derivative at one point, it projects a **secant line** through two 
historical points.

1. **Initialization:** Choose two initial guesses, $x_0$ and $x_1$ (they do not 
need to bracket the root).
2. **Secant Approximation:** Approximate the derivative using the slope between 
the two points:

$$f'(x_1) \approx \frac{f(x_1) - f(x_0)}{x_1 - x_0}$$


3. **Update Formula:** Substitute this finite-difference approximation into the Newton-Raphson update equation to find the next point $x_{i+1}$:

$$x_{i+1} = x_i - f(x_i) \frac{x_i - x_{i-1}}{f(x_i) - f(x_{i-1})}$$


4. **Iteration:** Discard the oldest point ($x_{i-1}$), retain the newest two, 
and loop until convergence.

* **Pros:** No derivative function $f'(x)$ required; faster than Bisection 
(superlinear convergence rate, roughly $\approx 1.62$).
* **Cons:** Still prone to divergence if the function behaves erratically or if 
$f(x_i) \approx f(x_{i-1})$.